In [1]:
from google.colab import drive
import sys
import torch

In [2]:
drive.mount('/content/drive')
sys.path.append(
"/content/drive/MyDrive/SupportAI"
)

Mounted at /content/drive


In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import json

In [4]:
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

path = "/content/drive/MyDrive/SupportAI/models"
topic_model = AutoModelForSequenceClassification.from_pretrained(f"{path}/topic")
sentiment_model = AutoModelForSequenceClassification.from_pretrained(f"{path}/sentiment")
intent_model = AutoModelForSequenceClassification.from_pretrained(f"{path}/intent")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
path1 = '/content/drive/MyDrive/SupportAI'
with open(f'{path1}/label_mappings.json', 'r') as f:
    label_mapping = json.load(f)

In [6]:
def predict(text, model, tokenizer):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=1)

    pred_id = torch.argmax(probs, dim=1).item()
    confidence = probs[0, pred_id].item()

    return {
        "label": model.config.id2label[pred_id],
        "confidence": confidence
    }

In [12]:
def load_models(path):

    tokenizer = AutoTokenizer.from_pretrained(
        "DeepPavlov/rubert-base-cased"
    )

    models = {
        "topic": AutoModelForSequenceClassification.from_pretrained(
            f"{path}/topic"
        ),
        "sentiment": AutoModelForSequenceClassification.from_pretrained(
            f"{path}/sentiment"
        ),
        "intent": AutoModelForSequenceClassification.from_pretrained(
            f"{path}/intent"
        ),
    }

    return tokenizer, models

In [9]:
def predict_all(text, models, tokenizer):
    results = {}

    for task, model in models.items():
        results[task] = predict(text, model, tokenizer)

    return results